In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [27]:
from broharness.llms.bedrock import bedrock, UserMessage, AIMessage, SystemMessage
from broharness.flows.skill_call import SkillCall
from broharness.flows.tool_call import ToolCall
from broharness.flows.tool_use import ToolUse
from broharness.flows.ask_user_question import AskUserQuestion
from broharness.flows.fail_recovery import FailRecovery
from broharness.flows.answer import Answer
from broflow import BaseTask, TaskRegistry, Flow
from pathlib import Path
import yaml
import sys
import subprocess
from broskill import SkillControl, ToolControl
from functools import partial
from broharness.toolblock import (
    tool_to_yaml, 
    load_skill_tool, 
    load_skill_extension_tool, 
    load_tool_tool, 
    ask_user_question_tool
)
from broharness.codeblock import parse_json_codeblock
from broharness.data_model import Process, State, LLMUse


ROOT = Path.cwd().resolve().parent
SKILL_DIR = ROOT / "skills"

sc = SkillControl(SKILL_DIR)
sc.list_skills()
tc = ToolControl(sc)

In [28]:
from broskill.processing.tool import to_args

In [29]:
tc.load_tool('read-file', 'scripts/read_file.py')

Tool(name='read_file', description="Read exactly one file's content, found via a glob pattern that must match a single file.", args=[Arg(name='pattern', type='string', description="Glob pattern, relative to the project root, that matches exactly one file (e.g. 'skills/tell-joke/references/dad-joke.md').", required=True)], path=WindowsPath('D:/study-on-agent/skills/read-file/scripts/read_file.py'))

In [30]:
TOOLS = dict(
    load_skill=sc.load_skill,
    load_skill_extension=sc.load_skill_extension,
    load_tool=tc.load_tool,
)

In [31]:
skill_call = SkillCall(name='skill-call', llm=bedrock, system_prompt='')
tool_call = ToolCall(name='tool-call', llm=bedrock, system_prompt='')
tool_use = ToolUse(name='tool-use', llm=bedrock, system_prompt='')
ask_user_question = AskUserQuestion(name='ask-user-question', llm=bedrock, system_prompt='')
fail_recovery = FailRecovery(name='fail-recovery', llm=bedrock, system_prompt='')
answer = Answer(name='answer', llm=bedrock, system_prompt='')

In [32]:
registry = TaskRegistry()
registry.register(Process.SKILL_CALL, skill_call)
registry.register(Process.TOOL_CALL, tool_call)
registry.register(Process.TOOL_USE, tool_use)
registry.register(Process.ASK_USER_QUESTION, ask_user_question)
registry.register(Process.FAIL_RECOVERY, fail_recovery)
registry.register(Process.ANSWER, answer)

flow = Flow(registry)

In [33]:
system_prompt = "You're Andy who is the best bro in the world. Always response in bro-tone with chill and mellow manner."

In [34]:
# this trigger load_skill with skill_name='tell-jokes'
# content = "tell me some jokes."
# this trigger ask_user_question
# content = "What's the capital of France?"
# this trigger nothing
# content = "1+1 is?"
# this trigger read_file
# content = "what is in skills/tell-joke/SKILL.md?"
content = "What's in `one-liner.md`"
# this trigger list_directory, shallow (top-level folders under skills/)
# content = "what folders are directly under skills/?"
# this trigger list_directory, recursive (everything under skills/)
# content = "list everything under skills/, including subfolders."

messages = [UserMessage(content)]
state = State(
    root=ROOT,
    skill_dir=SKILL_DIR,
    messages=messages,
    session_messages=messages.copy(),
    system_prompt=system_prompt or '',
    skill_control=sc,
    tool_control=tc,
    tools=TOOLS,
    session_tools=TOOLS.copy(),
    debug=messages.copy()
)

_ = flow.run(start=Process.SKILL_CALL, end=Process.END, state=state)

D:\study-on-agent\src\broharness\flows\skill_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'load_skill', 'input': {'skill_name': 'read-file'}}]
load_skill passed
auto-registered list_directory
auto-registered read_file
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'read_file', 'input': {'pattern': '**one-liner.md'}}]
invalid glob pattern: **one-liner.md (Invalid pattern: '**' can only be an entire path component)

D:\study-on-agent\src\broharness\flows\fail_recovery.py
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': 'read_file', 'input': {'pattern': '**one-liner.md'}}]
invalid glob pattern: **one-liner.md (Invalid pattern: '**' can only be an entire path component)

D:\study-on-agent\src\broharness\flows\fail_recovery.py
D:\study-on-agent\src\broharness\flows\tool_call.py
D:\study-on-agent\src\broharness\flows\tool_use.py
[{'name': '

In [35]:
state.messages

[{'role': 'user', 'content': [{'text': "What's in `one-liner.md`"}]},
 {'role': 'assistant',
  'content': [{'text': 'The file `one-liner.md` contains a style guide for writing one-liner jokes. It describes one-liners as single-sentence jokes that reframe ordinary things in a funny, understated way, emphasizing deadpan delivery and observations about everyday life rather than puns or name gags.'}]}]

In [36]:
print(state.messages[-1]['content'][0]['text'])

The file `one-liner.md` contains a style guide for writing one-liner jokes. It describes one-liners as single-sentence jokes that reframe ordinary things in a funny, understated way, emphasizing deadpan delivery and observations about everyday life rather than puns or name gags.


In [37]:
state.session_messages

[{'role': 'user', 'content': [{'text': "What's in `one-liner.md`"}]},
 {'role': 'assistant',
  'content': [{'text': 'The file `one-liner.md` contains a style guide for writing one-liner jokes. It describes one-liners as single-sentence jokes that reframe ordinary things in a funny, understated way, emphasizing deadpan delivery and observations about everyday life rather than puns or name gags.'}]}]

In [38]:
state.registered_skills

{'read-file': '# Read File\n\n## Instructions\n\n- If you already know the exact file the user means, use `scripts/read_file.py` with a\n  glob pattern narrow enough to match exactly that one file.\n- If the user names just a filename with no path (e.g. "what\'s in one-liner.md"), don\'t\n  guess its directory -- search for it anywhere in the project with `**/<filename>`\n  (e.g. `**/one-liner.md`). This isn\'t a guess, it\'s an exact-name search; if more than\n  one file shares that name, `read_file.py`\'s normal ambiguous-match handling applies.\n- If `scripts/read_file.py`\'s pattern matches more than one file, don\'t pick one\n  yourself even if it looks obvious -- call `ask_user_question` with the candidate\n  list from the error message and let the user choose.\n- If the user wants to see many files at once, or you aren\'t sure which single file they\n  mean, use `scripts/list_directory.py` first to see what matches a broader pattern, then\n  narrow down before reading.\n- `scrip

In [39]:
print('\n\n'.join(state.extension_skills.values()))

In [40]:
state.session_tools

{'load_skill': <bound method SkillControl.load_skill of <broskill.processing.skill.SkillControl object at 0x0000018B8F34E900>>,
 'load_skill_extension': <bound method SkillControl.load_skill_extension of <broskill.processing.skill.SkillControl object at 0x0000018B8F34E900>>,
 'load_tool': <bound method ToolControl.load_tool of <broskill.processing.tool.ToolControl object at 0x0000018B8F34E300>>}

In [41]:
state.error_message

''

In [42]:
state.debug

[{'role': 'user', 'content': [{'text': "What's in `one-liner.md`"}]},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "load_skill",\n      "input": {\n        "skill_name": "read-file"\n      }\n    }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 1058, 'outputTokens': 57, 'totalTokens': 1115}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "read_file",\n      "input": {\n        "pattern": "**one-liner.md"\n      }\n    }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 2377, 'outputTokens': 57, 'totalTokens': 2434}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "read_file",\n      "input": {\n        "pattern": "**one-liner.md"\n      }\n    }\n  ]\n}\n```'}],
  'usage': {'inputTokens': 2407, 'outputTokens': 57, 'totalTokens': 2464}},
 {'role': 'assistant',
  'content': [{'text': '```json\n{\n  "tool_use": [\n    {\n      "name": "list_direct

In [43]:
state.tool_results

[{'list_directory': 'DIAGRAM.md\nREADME.md\nskills\\read-file\\SKILL.md\nskills\\skill-call\\SKILL.md\nskills\\tell-joke\\references\\dad-joke.md\nskills\\tell-joke\\references\\knock-knock.md\nskills\\tell-joke\\references\\one-liner.md\nskills\\tell-joke\\references\\pun-joke.md\nskills\\tell-joke\\references\\riddle.md\nskills\\tell-joke\\SKILL.md\nskills\\tool-call\\SKILL.md\n'},
 {'read_file': '# One-liners (observational)\n\nA one-liner is a single sentence that reframes something ordinary in an\nunexpectedly funny way -- deadpan and understated rather than punchline-heavy,\ncloser to Steven Wright or Mitch Hedberg than a dad joke.\n\n## Characteristics\n\n- One sentence, sometimes two -- no setup/punchline split, the whole thing is\n  the joke.\n- The humor comes from a skewed, matter-of-fact observation about everyday life,\n  not wordplay -- notice something ordinary and state it in a way that reveals\n  its absurdity.\n- Deadpan delivery -- no exclamation, no "get it?", the f

In [44]:
for ts in state.tool_results:
    for k, v in ts.items():
        print(f'{k}: {v}')

list_directory: DIAGRAM.md
README.md
skills\read-file\SKILL.md
skills\skill-call\SKILL.md
skills\tell-joke\references\dad-joke.md
skills\tell-joke\references\knock-knock.md
skills\tell-joke\references\one-liner.md
skills\tell-joke\references\pun-joke.md
skills\tell-joke\references\riddle.md
skills\tell-joke\SKILL.md
skills\tool-call\SKILL.md

read_file: # One-liners (observational)

A one-liner is a single sentence that reframes something ordinary in an
unexpectedly funny way -- deadpan and understated rather than punchline-heavy,
closer to Steven Wright or Mitch Hedberg than a dad joke.

## Characteristics

- One sentence, sometimes two -- no setup/punchline split, the whole thing is
  the joke.
- The humor comes from a skewed, matter-of-fact observation about everyday life,
  not wordplay -- notice something ordinary and state it in a way that reveals
  its absurdity.
- Deadpan delivery -- no exclamation, no "get it?", the flatter the better.
- Doesn't rely on a pun or a name gag -- 

In [45]:
print(state.tool_results[0]['list_directory'])

DIAGRAM.md
README.md
skills\read-file\SKILL.md
skills\skill-call\SKILL.md
skills\tell-joke\references\dad-joke.md
skills\tell-joke\references\knock-knock.md
skills\tell-joke\references\one-liner.md
skills\tell-joke\references\pun-joke.md
skills\tell-joke\references\riddle.md
skills\tell-joke\SKILL.md
skills\tool-call\SKILL.md



In [46]:
state.usage

{'XXX_CALL': {'google.gemma-3-12b-it': {'input_tokens': 16016,
   'output_tokens': 378,
   'call_count': 7}},
 'ANSWER': {'google.gemma-3-12b-it': {'input_tokens': 1663,
   'output_tokens': 60,
   'call_count': 1}}}

In [47]:
def cal_usage(input, output, input_price, output_price, currency=1, session=1):
    mil = 1_000_000
    inputPrice = (input/mil)*input_price*currency*session
    outputPrice = (output/mil)*output_price*currency*session
    return inputPrice, outputPrice, inputPrice+outputPrice

In [51]:
cal_usage(16016+1663, 378+60, 0.09, 0.29)

(0.00159111, 0.00012702, 0.00171813)

In [55]:
cal_usage(16016+1663, 378+60, 0.09, 0.29, 34, 1)

(0.05409774, 0.004318679999999999, 0.05841642)

In [50]:
m_usage = {}
for m_type, m in state.usage.items():
    for _m, u in m.items():
        # if _m not in m_usage:
        #     m_usage[_m] = {"input_tokens": 0, "output_tokens": 0, "call_count": 0}
        print(f'{m_type} - {_m}: {u}')

XXX_CALL - google.gemma-3-12b-it: {'input_tokens': 16016, 'output_tokens': 378, 'call_count': 7}
ANSWER - google.gemma-3-12b-it: {'input_tokens': 1663, 'output_tokens': 60, 'call_count': 1}
